# **ACDC In-Domain Training — YOLOv11s**

*   Training YOLOv11s using COCO pretrained weights
*   Evaluating the trained model on the global ACDC
*   Performing weather-specific evaluations on fog, rain, and snow subsets


### **Mount drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### **Install Ultralytics**

In [ ]:
!pip install -q ultralytics

### **Import librairies**

In [ ]:
from pathlib import Path
import pandas as pd
import json
import yaml
import torch
import time
import cv2
import numpy as np
from ultralytics import YOLO

### **Path configuration & check GPU**

In [ ]:

PROJECT_ROOT = Path("/content/drive/MyDrive/Dissertation")

YOLO_ROOT = PROJECT_ROOT / "Datasets/processed/acdc_yolo"
RUNS_ROOT = PROJECT_ROOT / "Runs/yolo"

ACDC_YAML = YOLO_ROOT / "acdc.yaml"

print("YOLO_ROOT exists:", YOLO_ROOT.exists())
print("ACDC YAML exists:", ACDC_YAML.exists())
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

YOLO_ROOT exists: True
ACDC YAML exists: True
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


### **Classes & random seed**

In [ ]:
CLASS_NAMES = [
    "person",
    "bicycle",
    "car",
    "motorcycle",
    "bus",
    "truck"
]

RANDOM_SEED = 42

### **Check dataset folders**

In [ ]:
for split in ["train", "val", "test", "test_fog", "test_rain", "test_snow"]:
    img_dir = YOLO_ROOT / "images" / split
    label_dir = YOLO_ROOT / "labels" / split

    print(f"\n{split}")
    print("images:", img_dir.exists(), len(list(img_dir.glob("*"))) if img_dir.exists() else 0)
    print("labels:", label_dir.exists(), len(list(label_dir.glob("*"))) if label_dir.exists() else 0)


train
images: True 1620
labels: True 1620

val
images: True 540
labels: True 540

test
images: True 540
labels: True 540

test_fog
images: True 100
labels: True 100

test_rain
images: True 340
labels: True 340

test_snow
images: True 100
labels: True 100


### **Training**

In [ ]:
model = YOLO("yolo11s.pt")

train_results = model.train(
    data=str(ACDC_YAML),
    epochs=100,
    imgsz=640,
    batch=16,
    patience=10,
    seed=RANDOM_SEED,
    deterministic=True,
    pretrained=True,
    project=str(RUNS_ROOT),
    name="acdc_in_domain_yolo11s_seed42",
    exist_ok=True,
    plots=True
)

Ultralytics 8.4.50 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Dissertation/Datasets/processed/acdc_yolo/acdc.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=acdc_in_domain_yolo11s_seed42, nbs=64, nms=F

### **Load best model**

In [ ]:
RUN_DIR = RUNS_ROOT / "acdc_in_domain_yolo11s_seed42"
BEST_MODEL = RUN_DIR / "weights" / "best.pt"

print("Best model exists:", BEST_MODEL.exists())
print("Best model path:", BEST_MODEL)

model = YOLO(str(BEST_MODEL))

Best model exists: True
Best model path: /content/drive/MyDrive/Dissertation/Runs/yolo/acdc_in_domain_yolo11s_seed42/weights/best.pt


### **Global and weather-specific evaluation**

In [ ]:
eval_configs = {
    "global": YOLO_ROOT / "acdc.yaml",
    "fog": YOLO_ROOT / "fog_only.yaml",
    "rain": YOLO_ROOT / "rain_only.yaml",
    "snow": YOLO_ROOT / "snow_only.yaml",
}

results_rows = []

for condition, yaml_path in eval_configs.items():
    print(f"\nEvaluating ACDC {condition.upper()}...")

    metrics = model.val(
        data=str(yaml_path),
        split="test",
        imgsz=640,
        batch=16,
        plots=True,
        project=str(RUN_DIR / "evaluation"),
        name=f"test_{condition}",
        exist_ok=True
    )

    precision = float(metrics.box.mp)
    recall = float(metrics.box.mr)
    map50 = float(metrics.box.map50)
    map75 = float(metrics.box.map75)
    map5095 = float(metrics.box.map)

    f1 = 2 * precision * recall / (precision + recall + 1e-9)

    inference_ms = float(metrics.speed.get("inference", 0))
    fps = 1000 / inference_ms if inference_ms > 0 else None

    results_rows.append({
        "dataset": "ACDC",
        "model": "YOLOv11s",
        "experiment": "in_domain",
        "seed": RANDOM_SEED,
        "condition": condition,
        "mAP50-95": map5095,
        "mAP50": map50,
        "mAP75": map75,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "Inference_ms_per_image": inference_ms,
        "FPS": fps
    })

results_df = pd.DataFrame(results_rows)
results_df


Evaluating ACDC GLOBAL...
Ultralytics 8.4.50 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
val: Fast image access ✅ (ping: 0.6±0.2 ms, read: 204.8±38.1 MB/s, size: 459.2 KB)
val: Scanning /content/drive/.shortcut-targets-by-id/17lufoiYQdFrVMwNFeK91HKX-sBA0zN4V/Dissertation/Datasets/processed/acdc_yolo/labels/test.cache... 540 images, 7 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 540/540 151.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 34/34 6.2it/s 5.5s
                   all        540       3358      0.765      0.441      0.521      0.333
                person        214        556      0.888      0.336      0.489      0.262
               bicycle         73        117      0.513      0.316      0.313      0.187
                   car        521       2342      0.904      0.632      0.756      0.506
            motorcycle         62         77      0.756      0.273      0.344    

,dataset,model,experiment,seed,condition,mAP50-95,mAP50,mAP75,Precision,Recall,F1-score,Inference_ms_per_image,FPS
0,ACDC,YOLOv11s,in_domain,42,global,0.332924,0.521167,0.345546,0.765000,0.441276,0.559700,0.963798,1037.562229
1,ACDC,YOLOv11s,in_domain,42,fog,0.412664,0.627669,0.427248,0.858360,0.546958,0.668158,2.106268,474.773303
2,ACDC,YOLOv11s,in_domain,42,rain,0.308140,0.483562,0.319447,0.766111,0.405806,0.530571,1.796670,556.585328
3,ACDC,YOLOv11s,in_domain,42,snow,0.355982,0.559802,0.364799,0.724260,0.460557,0.563062,1.050349,952.064658


### **Save Results**

In [ ]:
results_csv = RUN_DIR / "acdc_yolo11s_in_domain_results_summary.csv"
results_json = RUN_DIR / "acdc_yolo11s_in_domain_results_summary.json"

results_df.to_csv(results_csv, index=False)

with open(results_json, "w") as f:
    json.dump(results_rows, f, indent=4)

print("Saved:")
print(results_csv)
print(results_json)

Saved:
/content/drive/MyDrive/Dissertation/Runs/yolo/acdc_in_domain_yolo11s_seed42/acdc_yolo11s_in_domain_results_summary.csv
/content/drive/MyDrive/Dissertation/Runs/yolo/acdc_in_domain_yolo11s_seed42/acdc_yolo11s_in_domain_results_summary.json


### **FPS Calcul**

In [ ]:
def get_image_paths_yolo(yolo_root, split_name):
    img_dir = yolo_root / "images" / split_name
    paths = []
    for ext in ["*.jpg", "*.jpeg", "*.png"]:
        paths.extend(img_dir.glob(ext))
    return sorted(paths)

def measure_yolo_fps_loaded_images(model, image_paths, imgsz=640, warmup=30):
    images = []

    for p in image_paths:
        img = cv2.imread(str(p))
        if img is not None:
            images.append(img)

    print("Loaded images:", len(images))

    # Warmup
    for img in images[:warmup]:
        _ = model.predict(
            source=img,
            imgsz=imgsz,
            verbose=False,
            device=0
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    times = []

    for img in images:
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start = time.perf_counter()

        _ = model.predict(
            source=img,
            imgsz=imgsz,
            verbose=False,
            device=0
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        end = time.perf_counter()
        times.append(end - start)

    avg_ms = np.mean(times) * 1000
    fps = 1000 / avg_ms

    return avg_ms, fps

split_mapping = {
    "global": "test",
    "fog": "test_fog",
    "rain": "test_rain",
    "snow": "test_snow",
}

fps_rows = []

for condition, split_name in split_mapping.items():
    print(f"\nMeasuring YOLO FPS for ACDC {condition.upper()}...")

    image_paths = get_image_paths_yolo(YOLO_ROOT, split_name)

    inference_ms, fps = measure_yolo_fps_loaded_images(
        model=model,
        image_paths=image_paths,
        imgsz=640,
        warmup=30
    )

    fps_rows.append({
        "dataset": "ACDC",
        "model": "YOLOv11s",
        "experiment": "in_domain",
        "seed": 42,
        "condition": condition,
        "Inference_ms_per_image": inference_ms,
        "FPS": fps
    })

fps_df = pd.DataFrame(fps_rows)
fps_df


Measuring YOLO FPS for ACDC GLOBAL...
Loaded images: 540

Measuring YOLO FPS for ACDC FOG...
Loaded images: 100

Measuring YOLO FPS for ACDC RAIN...
Loaded images: 340

Measuring YOLO FPS for ACDC SNOW...
Loaded images: 100


,dataset,model,experiment,seed,condition,Inference_ms_per_image,FPS
0,ACDC,YOLOv11s,in_domain,42,global,11.829795,84.532316
1,ACDC,YOLOv11s,in_domain,42,fog,11.745399,85.139722
2,ACDC,YOLOv11s,in_domain,42,rain,11.654375,85.804685
3,ACDC,YOLOv11s,in_domain,42,snow,11.945242,83.715342


In [ ]:
results_json = RUN_DIR / "acdc_yolo11s_in_domain_results_summary.json"
results_csv = RUN_DIR / "acdc_yolo11s_in_domain_results_summary.csv"

with open(results_json, "r") as f:
    results_data = json.load(f)

results_df = pd.DataFrame(results_data)

for _, fps_row in fps_df.iterrows():
    condition = fps_row["condition"]

    results_df.loc[
        results_df["condition"] == condition,
        "Inference_ms_per_image"
    ] = fps_row["Inference_ms_per_image"]

    results_df.loc[
        results_df["condition"] == condition,
        "FPS"
    ] = fps_row["FPS"]

results_df.to_csv(results_csv, index=False)

with open(results_json, "w") as f:
    json.dump(results_df.to_dict(orient="records"), f, indent=4)

results_df

,dataset,model,experiment,seed,condition,mAP50-95,mAP50,mAP75,Precision,Recall,F1-score,Inference_ms_per_image,FPS
0,ACDC,YOLOv11s,in_domain,42,global,0.332924,0.521167,0.345546,0.765000,0.441276,0.559700,11.829795,84.532316
1,ACDC,YOLOv11s,in_domain,42,fog,0.412664,0.627669,0.427248,0.858360,0.546958,0.668158,11.745399,85.139722
2,ACDC,YOLOv11s,in_domain,42,rain,0.308140,0.483562,0.319447,0.766111,0.405806,0.530571,11.654375,85.804685
3,ACDC,YOLOv11s,in_domain,42,snow,0.355982,0.559802,0.364799,0.724260,0.460557,0.563062,11.945242,83.715342
